#### solar panels are mapped in great detail. they show up as multiple assets/ways, despite being on one farm. this is why deduplication is helpful

In [1]:
import pandas as pd
from scipy.spatial import KDTree
import numpy as np

df = pd.read_csv("data/maine_all_assets.csv")
solar = df[df["asset_type"] == "energy.generation.solar_farm"]
coords = solar[["lat", "lon"]].values
tree = KDTree(coords)
pairs = tree.query_pairs(200 / 111320)  # 200m in degrees
print(f"Solar farms within 200m of another: {len(pairs)} pairs")
print(f"Total solar farms: {len(solar)}")

Solar farms within 200m of another: 20633 pairs
Total solar farms: 737


In [1]:
import pandas as pd
import json

df = pd.read_csv("data/central-america_all_assets_collapsed.csv")

solar_inf = df[df["asset_type"] == "energy.generation.solar_facility_inferred"]

largest = solar_inf.sort_values("n_merged_parts", ascending=False).iloc[0]

print("Cluster size:", largest["n_merged_parts"])

tags = json.loads(largest["osm_tags"])

print("First 20 member IDs:", tags["member_ids"][:20])

Cluster size: 11905
First 20 member IDs: ['osm_way_1159261966', 'osm_way_1340119214', 'osm_way_1340120827', 'osm_way_1340261132', 'osm_way_1340108836', 'osm_way_1340108842', 'osm_way_1340108843', 'osm_way_1340108841', 'osm_way_1340108839', 'osm_way_1340108838', 'osm_way_1340108840', 'osm_way_1340108832', 'osm_way_1340108845', 'osm_way_1340108844', 'osm_way_1340108846', 'osm_way_1340108847', 'osm_way_1340108848', 'osm_way_1340108849', 'osm_way_1340330630', 'osm_way_1340312057']


In [ ]:
df = pd.read_csv("data/maine_all_assets_collapsed.csv")

print(df[df.asset_type.str.contains("substation")].shape)

(386, 10)


In [1]:
import pandas as pd

csv_df = pd.read_csv("data/maine_all_assets_collapsed.csv")
pq_df = pd.read_parquet("data/maine_all_assets_collapsed.parquet")

print("CSV rows:", len(csv_df))
print("Parquet rows:", len(pq_df))
print("Same columns:", list(csv_df.columns) == list(pq_df.columns))

cols = [
    "asset_id",
    "asset_type",
    "lat",
    "lon",
    "solar_facility_type",
    "inferred_confidence",
    "n_merged_parts",
]

csv_cmp = csv_df[cols].sort_values("asset_id").reset_index(drop=True)
pq_cmp = pq_df[cols].sort_values("asset_id").reset_index(drop=True)

print("Key fields equal:", csv_cmp.equals(pq_cmp))

CSV rows: 678
Parquet rows: 678
Same columns: True
Key fields equal: False


In [2]:
import pandas as pd

csv_df = pd.read_csv("data/maine_all_assets_collapsed.csv")
pq_df = pd.read_parquet("data/maine_all_assets_collapsed.parquet")

cols = [
    "asset_id",
    "asset_type",
    "lat",
    "lon",
    "solar_facility_type",
    "inferred_confidence",
    "n_merged_parts",
]

csv_cmp = csv_df[cols].sort_values("asset_id").reset_index(drop=True).copy()
pq_cmp = pq_df[cols].sort_values("asset_id").reset_index(drop=True).copy()

# Normalize likely trouble spots
for c in ["solar_facility_type", "inferred_confidence"]:
    csv_cmp[c] = csv_cmp[c].fillna("").astype(str)
    pq_cmp[c] = pq_cmp[c].fillna("").astype(str)

csv_cmp["lat"] = csv_cmp["lat"].astype(float).round(8)
csv_cmp["lon"] = csv_cmp["lon"].astype(float).round(8)
pq_cmp["lat"] = pq_cmp["lat"].astype(float).round(8)
pq_cmp["lon"] = pq_cmp["lon"].astype(float).round(8)

csv_cmp["n_merged_parts"] = csv_cmp["n_merged_parts"].astype(int)
pq_cmp["n_merged_parts"] = pq_cmp["n_merged_parts"].astype(int)

print("Key fields equal:", csv_cmp.equals(pq_cmp))

# If still false, show the first mismatch
diff_mask = (csv_cmp != pq_cmp).any(axis=1)
if diff_mask.any():
    idx = diff_mask.idxmax()
    print("\nCSV row:")
    print(csv_cmp.loc[idx])
    print("\nParquet row:")
    print(pq_cmp.loc[idx])

Key fields equal: True


In [ ]:
from utils.io_utils import load_asset_table
from deduplication import Deduplicator

df = load_asset_table("data/maine_all_assets_collapsed.parquet")

dedup = Deduplicator(distance_threshold_m=200)
df_clean, df_removed = dedup.run(df)

print("Input rows:", len(df))
print("Deduped rows:", len(df_clean))
print("Removed rows:", len(df_removed))
print(df_clean["asset_type"].value_counts())

Deduplication complete:
  Kept:    660
  Removed: 18 (2.7%)

  Removed by asset type:
asset_type
energy.generation.wind_farm               9
energy.generation.generator               7
energy.distribution.substation_untyped    1
energy.generation.power_plant             1
Input rows: 678
Deduped rows: 660
Removed rows: 18
asset_type
energy.generation.wind_farm                  419
energy.distribution.substation_untyped        84
energy.distribution.substation                60
energy.generation.power_plant                 41
energy.generation.solar_farm                  22
energy.generation.generator                   18
energy.transmission.substation                13
energy.generation.solar_facility_inferred      3
Name: count, dtype: int64


In [5]:
import pandas as pd

df = pd.read_csv("data/central-america_all_assets_collapsed.csv")

# make osm_tags consistent with solar_collapse outputs
import json
if "osm_tags" in df.columns:
    df["osm_tags"] = df["osm_tags"].apply(
        lambda x: x if isinstance(x, str) else json.dumps(x)
    )

df.to_parquet("data/central_america_all_assets.parquet", index=False)

print("Rows:", len(df))

Rows: 4621


In [8]:
import pandas as pd

orig = pd.read_parquet("../data/PIPELINE/01-extracted-assets/maine_all_assets_collapsed.parquet")
slim = pd.read_parquet("../data/PIPELINE/01-extracted-assets/maine_all_assets_slim_collapsed.parquet")

orig_ids = set(orig["asset_id"])
slim_ids = set(slim["asset_id"])

missing = orig[orig["asset_id"].isin(orig_ids - slim_ids)].copy()

print("Missing count:", len(missing))
print("\nMissing by asset_type:")
print(missing["asset_type"].value_counts())

print("\nSample missing rows:")
print(missing[["asset_id", "asset_type", "lat", "lon"]].head(25).to_string(index=False))

Missing count: 42

Missing by asset_type:
asset_type
energy.distribution.substation_untyped       24
energy.distribution.substation                9
energy.generation.power_plant                 5
energy.transmission.substation                2
energy.generation.generator                   1
energy.generation.solar_facility_inferred     1
Name: count, dtype: int64

Sample missing rows:
         asset_id                             asset_type       lat        lon
osm_way_238220158 energy.distribution.substation_untyped 43.363628 -70.467356
osm_way_238563717         energy.distribution.substation 43.407030 -70.710861
osm_way_238563718 energy.distribution.substation_untyped 43.148570 -70.678666
osm_way_238567340         energy.distribution.substation 45.101282 -70.353305
osm_way_238567344         energy.distribution.substation 43.620844 -70.329854
osm_way_238567347         energy.distribution.substation 43.666023 -70.596007
osm_way_238567348         energy.distribution.substation 43.63104

In [3]:
import pandas as pd

df = pd.read_csv("../outputs/caadd_labels_maine_v1.csv")
print(df["accessibility_label"].value_counts())
print("Majority baseline:", df["accessibility_label"].value_counts(normalize=True).max())

accessibility_label
partially_accessible    121
mostly_accessible         8
Name: count, dtype: int64
Majority baseline: 0.937984496124031


In [ ]:
# test_single_tile.py
import pystac_client
import planetary_computer
import rasterio
from rasterio.crs import CRS
from rasterio.warp import transform_bounds
import numpy as np

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

bbox = (-70.55, 43.61, -70.45, 43.71)

# --- Sentinel-2 ---
print("Testing Sentinel-2...")
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2021-01-01/2024-12-31",
    limit=50,
)
items = list(search.items())
items.sort(key=lambda x: x.properties.get("eo:cloud_cover", 100))
clean = [i for i in items if i.properties.get("eo:cloud_cover", 100) < 20]
print(f"  Found {len(clean)} clean scenes")

item = planetary_computer.sign(clean[0])
print(f"  Using: {item.id}")

href = item.assets["B04"].href
with rasterio.open(href) as src:
    bbox_native = transform_bounds(
        CRS.from_epsg(4326), src.crs,
        bbox[0], bbox[1], bbox[2], bbox[3],
    )
    window = rasterio.windows.from_bounds(*bbox_native, transform=src.transform)
    data = src.read(1, window=window)
    print(f"  B04 shape: {data.shape}")
    print(f"  B04 min/max: {data.min()} / {data.max()}")

# --- Sentinel-1 ---
print("\nTesting Sentinel-1...")
search = catalog.search(
    collections=["sentinel-1-grd"],
    bbox=bbox,
    datetime="2021-01-01/2024-12-31",
    limit=10,
)
items = list(search.items())
if items:
    item = planetary_computer.sign(items[0])
    print(f"  Using: {item.id}")
    print(f"  Assets: {[k for k in item.assets.keys() if k in ['VV', 'VH']]}")
    if "VV" in item.assets:
        href = item.assets["VV"].href
        with rasterio.open(href) as src:
            bbox_native = transform_bounds(
                CRS.from_epsg(4326), src.crs,
                bbox[0], bbox[1], bbox[2], bbox[3],
            )
            window = rasterio.windows.from_bounds(*bbox_native, transform=src.transform)
            data = src.read(1, window=window)
            print(f"  VV shape: {data.shape}")
            print(f"  VV min/max: {data.min():.3f} / {data.max():.3f}")
else:
    print("  No Sentinel-1 scenes found")

# --- Full STACImageryFetcher test ---
print("\nTesting STACImageryFetcher end-to-end...")
import pandas as pd
from curation.stac_imagery import STACImageryFetcher

df = pd.DataFrame([{
    "asset_id":   "test_001",
    "asset_type": "energy.distribution.substation_untyped",
    "lat":        43.66,
    "lon":        -70.25,
}])

fetcher = STACImageryFetcher(
    buffer_m=300,
    modalities=["sentinel2_ms"],
    temporal_stack=False,
)
results = fetcher.fetch_all(df)
r = results[0]
print(f"  status:      {r.status}")
print(f"  error_msg:   {r.error_msg}")
print(f"  image shape: {r.image_shape}")
print(f"  n_bands:     {r.n_bands}")
print(f"  image_date:  {r.image_date}")

Testing Sentinel-2...
  Found 194 clean scenes
  Using: S2B_MSIL2A_20231001T153559_R111_T19TCJ_20231001T225252
  B04 shape: (1125, 826)
  B04 min/max: 1106 / 13616

Testing Sentinel-1...
  Using: S1A_IW_GRDH_1SDV_20241227T223539_20241227T223604_057184_07084A
  Assets: []

Testing STACImageryFetcher end-to-end...


ModuleNotFoundError: No module named 'curation'

In [5]:
import numpy as np
tile = np.load("../data/curated_datasets/dataset_maine_stac_v1/images/osm_way_143617179_stac_sentinel2_ms+sentinel1+landsat_thermal.npy")
print(f"shape: {tile.shape}")
print(f"thermal band (index 9) min/max/mean: {tile[9].min():.1f} / {tile[9].max():.1f} / {tile[9].mean():.1f}")

shape: (10, 60, 60)
thermal band (index 9) min/max/mean: 0.0 / 0.0 / 0.0


In [6]:
# test_landsat.py
import pystac_client
import planetary_computer

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

bbox = (-70.55, 43.61, -70.45, 43.71)

search = catalog.search(
    collections=["landsat-c2-l2"],
    bbox=bbox,
    datetime="2021-01-01/2024-12-31",
    limit=5,
)
items = list(search.items())
print(f"Found {len(items)} Landsat scenes")

if items:
    items.sort(key=lambda x: x.properties.get("eo:cloud_cover", 100))
    item = planetary_computer.sign(items[0])
    print(f"Using: {item.id}")
    print(f"Cloud: {item.properties.get('eo:cloud_cover')}%")
    print(f"Assets: {[k for k in item.assets.keys()]}")
    print(f"ST_B10 present: {'ST_B10' in item.assets}")

Found 386 Landsat scenes
Using: LE07_L2SP_011030_20230528_02_T1
Cloud: 0.0%
Assets: ['qa', 'ang', 'red', 'blue', 'drad', 'emis', 'emsd', 'lwir', 'trad', 'urad', 'atran', 'cdist', 'green', 'nir08', 'swir16', 'swir22', 'mtl.txt', 'mtl.xml', 'cloud_qa', 'mtl.json', 'qa_pixel', 'qa_radsat', 'atmos_opacity', 'tilejson', 'rendered_preview']
ST_B10 present: False


In [8]:
# test_landsat3.py
import pystac_client
import planetary_computer

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

bbox = (-70.55, 43.61, -70.45, 43.71)

search = catalog.search(
    collections=["landsat-c2-l2"],
    bbox=bbox,
    datetime="2021-01-01/2024-12-31",
    query={"platform": {"in": ["landsat-8", "landsat-9"]}},
    limit=5,
)
items = list(search.items())
items.sort(key=lambda x: x.properties.get("eo:cloud_cover", 100))
item = planetary_computer.sign(items[0])

print(f"ID: {item.id}")
print(f"Platform: {item.properties.get('platform')}")
print(f"All asset keys:")
for k in sorted(item.assets.keys()):
    print(f"  {k}")

ID: LC08_L2SP_012030_20221103_02_T1
Platform: landsat-8
All asset keys:
  ang
  atran
  blue
  cdist
  coastal
  drad
  emis
  emsd
  green
  lwir11
  mtl.json
  mtl.txt
  mtl.xml
  nir08
  qa
  qa_aerosol
  qa_pixel
  qa_radsat
  red
  rendered_preview
  swir16
  swir22
  tilejson
  trad
  urad


In [9]:
import numpy as np
tile = np.load("../data/curated_datasets/dataset_maine_stac_v1/images/osm_way_168012266_stac_sentinel2_ms+sentinel1+landsat_thermal.npy")
valid = np.any(tile[:7] != 0, axis=0).mean()
print(f"Valid pixel ratio: {valid:.3f}")

Valid pixel ratio: 0.000
